# Filtering rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/y-rosenthal/yr-pre-ml/blob/main/jb/chapters/10-filtering-rows.ipynb)

A manager rarely wants every row. "Show me the big orders." "How many Electronics sales had a discount?" "List
the sales of the North stores." Each question keeps only the rows that match a condition. This is called
**filtering**, and pandas does it with the Boolean masks that we met for arrays. At the end, we use the same idea
to change the values of the matching rows, and we learn the one safe way to do it.

**In this chapter, you will learn to:**

- make a mask from a comparison on a column, and keep the rows where it is `True`;
- count the rows that match a condition;
- combine conditions with `&`, `|` and `~`, and test for several values with `isin`;
- filter rows and choose columns in one step with `loc`;
- change the values of the matching rows with `df.loc[mask, "col"] = value`, and avoid the chained-assignment trap.

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/y-rosenthal/yr-pre-ml/main/data/"

We use the sales of a store chain: 2,000 sales in one year, one row for each sale (see the chapter
*Reading and describing data*).

In [ ]:
sales = pd.read_csv(DATA_URL + "sales.csv")
sales.head()

## A comparison gives a mask

Which sales had more than 10 units? We compare the whole column with 10:

In [ ]:
many_units = sales["units"] > 10
many_units.head()

The result is a Series of `True` and `False`, one for each row, with the same index as `sales`. Compare it with
the table above: the rows 2 and 4 have 11 and 13 units, so they are `True`; the others have 10 units or fewer.
Its dtype is `bool`.

This is a **mask**, exactly as for NumPy arrays in the chapter *Comparisons and Boolean masks*. The only difference
is that a pandas mask also carries the row labels.

## Keeping the rows that match

Put the mask inside square brackets, and pandas keeps the rows where the mask is `True`:

In [ ]:
big_sales = sales[sales["units"] > 10]
big_sales.head()

Read `sales[sales["units"] > 10]` as "the rows of `sales` where `units` is above 10". The inner part is the mask
from above. Every row of `big_sales` has more than 10 units.

Look at the index: 2, 4, 11, 47, ... The rows keep their labels from `sales`, as they did after sorting in the
chapter *Selecting with loc and iloc*. So `big_sales.iloc[0]` is the row with the label 2.

## Counting the rows that match

How many sales had more than 10 units? There are two ways:

In [ ]:
print(many_units.sum())
print(len(big_sales))

Both say 75. `many_units.sum()` adds up the mask: `True` counts as 1 and `False` as 0, so the sum is the number of
`True` values. `len(big_sales)` counts the rows of the filtered table. Use the first one when you need only the
number.

> **Try it:** `many_units.mean()` gives the **share** of the sales with more than 10 units. What share is it, as
> a percentage?

## Conditions on text

A condition can compare a text column too. Here are the sales of store S1:

In [ ]:
s1_sales = sales[sales["store"] == "S1"]
len(s1_sales)

Store S1 made 287 of the 2,000 sales. Note the **two** equal signs: `==` compares, and `=` assigns.

> **Common mistake:** writing `sales[sales["store"] = "S1"]` with one `=`. Python stops with a `SyntaxError`
> before it runs the cell. And text must match exactly: `sales["store"] == "s1"` (a small "s") matches no row
> at all, without any error.

## Two conditions at once: `&`

Which Electronics sales had a discount? We need two conditions, and both must be true. As for arrays, we put each
condition in parentheses and join them with `&` ("and"):

In [ ]:
electronics_discount = (sales["category"] == "Electronics") & (sales["discount"] > 0)
electronics_discount.sum()

154 sales were Electronics sales with a discount. (Colab shows the sum as `np.int64(154)`, a NumPy whole number,
as for arrays.)

The parentheses are needed. Python computes `&` before `==` and `>`, so without them it would try to compute
`"Electronics" & sales["discount"]` first, and stop with an error.

> **Common mistake:** writing `and` instead of `&`. Python's `and` works on one `True` or `False`, not on a
> whole column, so pandas stops with `ValueError: The truth value of a Series is ambiguous`. With masks, always use
> `&`, `|` and `~`.

## Or, and not: `|` and `~`

`|` means "or": the row is kept when at least one condition is true. `~` means "not": it turns every `True` into
`False` and every `False` into `True`.

In [ ]:
s1_or_s2 = (sales["store"] == "S1") | (sales["store"] == "S2")
not_grocery = ~(sales["category"] == "Grocery")
print(s1_or_s2.sum())
print(not_grocery.sum())

556 sales were made in store S1 or store S2 (287 + 269). 1,206 sales were not Grocery sales: Clothing, Home and
Electronics together.

## Several values: `isin`

To keep the rows whose value is one of several, a chain of `|` gets long. `isin` takes a list of values:

In [ ]:
three_stores = sales["store"].isin(["S1", "S2", "S5"])
three_stores.sum()

`isin` gives a mask that is `True` where the store is `S1`, `S2` or `S5`: 817 sales. It is the same as
`(sales["store"] == "S1") | (sales["store"] == "S2") | (sales["store"] == "S5")`, but shorter and easier to read.

## Rows and columns at once with `loc`

Often we want the matching rows, but only some of the columns. In the chapter *Selecting with loc and iloc*, we
wrote `df.loc[rows, columns]` with labels. The row place also accepts a **mask**:

In [ ]:
sales.loc[sales["units"] > 10, ["date", "store", "units"]].head()

The rows with more than 10 units, and only the three columns that we named. With one column name instead of a
list, we get a Series, and we can summarize it: `sales.loc[sales["units"] > 10, "price"].mean()` is the average
price in the big sales.

> **Try it:** compare the average price of the sales with more than 10 units and of the sales with 10 units or
> fewer. Are big sales usually cheap products?

## Changing the matching rows

`loc` can also **change** the matching rows. A manager asks: "What if store S1 had given every customer a 50%
discount?" We try it on a copy, `sales2`, so that `sales` keeps the real data:

In [ ]:
sales2 = sales.copy()
sales2.loc[sales2["store"] == "S1", "discount"] = 0.5
sales2.head()

Row 0 is a sale of store S1, and its discount is now 0.5. The other rows of the first five are from other stores,
and they did not change.

Why `copy()`? `sales2 = sales` would **not** make a new table: it would give the same table a second name, as with
lists in Python, and a change through `sales2` would change `sales` too. `copy()` makes a separate table, as it did
for arrays in the chapter *Indexing, slicing and arithmetic*.

Let's check both tables:

In [ ]:
print((sales2["discount"] == 0.5).sum())
print((sales["discount"] == 0.5).sum())

All 287 sales of store S1 now have a discount of 0.5 in `sales2`, and `sales` has none. This is **the rule** for
changing values:

> **Note:** to change the rows that match a condition, write `df.loc[mask, "col"] = value`: the mask, the
> column and the new value, in **one** step.

## The chained-assignment trap

Many people first try something like this (it is wrong, so we do not run it):

```python
sales2[sales2["store"] == "S1"]["discount"] = 0.5     # wrong: does not change sales2
```

It looks reasonable, but it has two steps. First, `sales2[sales2["store"] == "S1"]` makes a **new** table, with
only the rows of store S1. Then `["discount"] = 0.5` changes the column of that new table. The new table has no
name, so it is thrown away at once, and `sales2` stays as it was. This is called **chained assignment**: two
selections in a chain, then `=`.

In pandas 3, chained assignment **never** changes `sales2`, and pandas shows a warning. In pandas 2 it does not
change `sales2` here either, but the same idea in the other order, `sales2["discount"][mask] = 0.5`, does change
it (with a warning). So code that seems to work in pandas 2 stops working in pandas 3, and only a warning tells
you. The warnings also differ between the versions, which is why we show this code without running it.

The fix is always the same: use `loc`, with the mask and the column in one pair of brackets,
`sales2.loc[sales2["store"] == "S1", "discount"] = 0.5`.

## Summary

- A comparison on a column, such as `sales["units"] > 10`, gives a **mask**: a Boolean Series with one
  `True`/`False` for each row.
- `sales[mask]` keeps the rows where the mask is `True`; they keep their labels.
- `mask.sum()` counts the matching rows (and `len(sales[mask])` too); `mask.mean()` gives their share.
- Combine conditions with `&` (and), `|` (or), `~` (not), with parentheses around each condition:
  `(sales["category"] == "Electronics") & (sales["discount"] > 0)`.
- `sales["store"].isin(["S1", "S2"])` is `True` where the value is one of the list.
- `sales.loc[mask, ["date", "store", "units"]]` filters rows and chooses columns at once.
- `df.loc[mask, "col"] = value` changes the matching rows. Work on a copy (`sales2 = sales.copy()`) to keep the
  original. Never chain two selections before `=`.

## Practice questions

### Question 1 (Easy)

How many sales were in the Electronics category?

<details class="yr-answer">
<summary>Answer</summary>

Compare the column with `"Electronics"`, and add up the mask. There were 287 Electronics sales.

```python
(sales["category"] == "Electronics").sum()
```

</details>

In [ ]:
# Write your answer here

### Question 2 (Easy)

Show the sales with a price above $500. How many are there?

<details class="yr-answer">
<summary>Answer</summary>

Filter with the mask, then count the rows of the result. There are 61 such sales, all of them Electronics.

```python
expensive = sales[sales["price"] > 500]
print(expensive.head())
print(len(expensive))
```

</details>

In [ ]:
# Write your answer here

### Question 3 (Easy)

Say in words which sales `sales[~(sales["discount"] > 0)]` keeps. How many are there?

<details class="yr-answer">
<summary>Answer</summary>

`sales["discount"] > 0` is `True` for the sales with a discount, and `~` turns it around: the sales with **no**
discount. There are 1,018 of them. `sales["discount"] == 0` gives the same rows.

```python
no_discount = sales[~(sales["discount"] > 0)]
len(no_discount)
```

</details>

In [ ]:
# Write your answer here

### Question 4 (Medium)

How many sales in the West region had at least 10 units?

<details class="yr-answer">
<summary>Answer</summary>

Both conditions must be true, so join them with `&`, each in parentheses. "At least 10" is `>= 10`. 31 sales
in the West region had at least 10 units.

```python
west_big = (sales["region"] == "West") & (sales["units"] >= 10)
west_big.sum()
```

</details>

In [ ]:
# Write your answer here

### Question 5 (Medium)

Show the Clothing and Home sales of store S3. Use `isin` for the two categories. How many are there?

<details class="yr-answer">
<summary>Answer</summary>

`isin` handles the two categories, and `&` adds the store. `sales["category"].isin(...)` needs no parentheses of
its own, because it is a method call, not a comparison. Store S3 made 120 Clothing and Home sales.

```python
s3_clothing_home = sales[(sales["store"] == "S3") & sales["category"].isin(["Clothing", "Home"])]
print(s3_clothing_home.head())
print(len(s3_clothing_home))
```

</details>

In [ ]:
# Write your answer here

### Question 6 (Medium)

Show only the date, the category and the price of the sales with a price above $400 (the first five rows).

<details class="yr-answer">
<summary>Answer</summary>

Use `loc` with the mask in the row place and a list of columns in the column place.

```python
sales.loc[sales["price"] > 400, ["date", "category", "price"]].head()
```

</details>

In [ ]:
# Write your answer here

### Question 7 (Hard)

What share of the sales of store S8 had a discount? First keep the sales of store S8, then use a mask on them.

<details class="yr-answer">
<summary>Answer</summary>

`mean()` of a mask gives the share of `True` values. About 50% of the sales of store S8 had a discount.

```python
s8 = sales[sales["store"] == "S8"]
(s8["discount"] > 0).mean()
```

</details>

In [ ]:
# Write your answer here

### Question 8 (Hard)

In a copy of `sales`, give every Electronics sale in December a discount of 0.2. How many rows did you change?
(Hint: the dates are text such as `"2025-12-05"`, and text in this form can be compared: `"2025-12-05" >=
"2025-12-01"` is `True`.)

<details class="yr-answer">
<summary>Answer</summary>

Make the copy first. The December sales are the ones whose date is `"2025-12-01"` or later. Then change the
matching rows with `loc`, in one step. The mask matches 19 sales, so `loc` set the discount of 19 rows.

```python
sales3 = sales.copy()
december_electronics = (sales3["category"] == "Electronics") & (sales3["date"] >= "2025-12-01")
sales3.loc[december_electronics, "discount"] = 0.2
print(december_electronics.sum())
print(sales3.loc[december_electronics, ["date", "category", "discount"]].head())
```

</details>

In [ ]:
# Write your answer here

### Question 9 (Challenge)

For each store, count the sales with more than 10 units. Which store had the most? Use a loop over the store
names, which `sorted(sales["store"].unique())` gives in order.

<details class="yr-answer">
<summary>Answer</summary>

For each store, make a mask with two conditions, and add it up. Store S1 had the most sales with more than 10
units: 16.

```python
for store in sorted(sales["store"].unique()):
    count = ((sales["store"] == store) & (sales["units"] > 10)).sum()
    print(store, count)
```

</details>

In [ ]:
# Write your answer here